# Assignment 2: Strengthened Text Classification, QA, and Insight Notebook

This notebook upgrades the project with:
- dataset QA visibility and duplicate-safe modeling inputs;
- a fixed 80/20 stratified holdout for demo visuals;
- 5-fold, 3-repeat repeated cross-validation for stronger evidence;
- an extra sparse baseline (`LinearSVC`);
- corrected TwitterRoBERTa preprocessing from the model card (`@user`, `http`);
- permutation testing, a learning curve, SHAP, and explicit error analysis;
- exported artifacts for report/README synchronization.


## 0. Google Colab & GitHub Sync
Recommended official path: run this notebook on Google Colab with a T4 GPU. The setup cell below supports read-only clone mode by default and optional GitHub push-back when secrets are provided.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
github_token = None
repo_owner = 'Krishna200608'
repo_name = 'BDA-MDD-Reddit-NLP'
git_user_name = 'Krishna200608'
git_user_email = 'krishnasikheriya001@gmail.com'


def run_command(command: list[str]) -> None:
    subprocess.run(command, check=True)


if IS_COLAB:
    from google.colab import userdata

    def get_secret(name: str) -> str | None:
        try:
            value = userdata.get(name)
            return value if value else None
        except Exception:
            return None

    github_token = get_secret('GITHUB_TOKEN') or os.getenv('GITHUB_TOKEN')

    clone_url = (
        f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"
        if github_token
        else f"https://github.com/{repo_owner}/{repo_name}.git"
    )

    repo_dir = Path(repo_name).resolve()
    if not repo_dir.exists():
        run_command(['git', 'clone', clone_url])

    if git_user_name and git_user_email:
        run_command(['git', 'config', '--global', 'user.name', git_user_name])
        run_command(['git', 'config', '--global', 'user.email', git_user_email])

    notebook_dir = repo_dir / 'notebooks'
    os.chdir(notebook_dir)
    os.environ['COLAB_REPO_DIR'] = str(repo_dir)

    run_command([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'shap',
        'wordcloud',
        'vaderSentiment',
        'schedule',
    ])

    print(f'Colab repo ready at: {repo_dir}')
    print(f'Working directory: {Path.cwd()}')
    print('Installed notebook dependencies for Colab.')
    if github_token:
        print('GitHub token detected: pushing updated artifacts back to GitHub is available.')
    else:
        print('No GitHub token detected: read-only clone mode is active.')
else:
    print('Running on local environment.')

## 1. Imports, Configuration, and Paths

In [ ]:
import hashlib
import os
import re
import sys
import warnings
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
from IPython.display import display
from tqdm.notebook import tqdm
from wordcloud import WordCloud

from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    learning_curve,
    permutation_test_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline

from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
tqdm.pandas()
pd.set_option('display.max_columns', None)

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = Path(os.environ['COLAB_REPO_DIR']).resolve() if os.environ.get('COLAB_REPO_DIR') else (CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR)
DATA_DIR = REPO_ROOT / 'data' / 'processed'
DATASET_PATH = DATA_DIR / 'reddit_mdd_cleaned.csv'
DATASET_SUMMARY_PATH = DATA_DIR / 'dataset_summary.csv'
RESULTS_SUMMARY_PATH = DATA_DIR / 'results_summary.csv'
ERROR_ANALYSIS_PATH = DATA_DIR / 'error_analysis_holdout.csv'
TOP_TOKENS_PATH = DATA_DIR / 'top_tokens_by_class.csv'

LABEL_ORDER = ['Control', 'Moderate MDD', 'Severe Ideation']
LABEL_TO_TARGET = {'Control': 0, 'Moderate MDD': 1, 'Severe Ideation': 2}
TARGET_TO_LABEL = {v: k for k, v in LABEL_TO_TARGET.items()}
SCORING = {'accuracy': 'accuracy', 'macro_f1': 'f1_macro', 'weighted_f1': 'f1_weighted'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

IS_COLAB = 'google.colab' in sys.modules
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'CPU'


def choose_roberta_batch_size(gpu_name: str, cuda_available: bool) -> int:
    if not cuda_available:
        return 16
    if 'T4' in gpu_name.upper():
        return 64
    return 32


ROBERTA_BATCH_SIZE = choose_roberta_batch_size(GPU_NAME, CUDA_AVAILABLE)
RSKF = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

PROGRESS_BAR_FORMAT = '{desc:<36}{percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'


def progress_bar(*args, colour: str = '#2563eb', leave: bool = True, **kwargs):
    return tqdm(
        *args,
        dynamic_ncols=True,
        leave=leave,
        colour=colour,
        bar_format=PROGRESS_BAR_FORMAT,
        **kwargs,
    )


def index_rows(values: object, indices: np.ndarray):
    if isinstance(values, (pd.Series, pd.DataFrame)):
        return values.iloc[indices]
    return values[indices]


def evaluate_cv_with_progress(
    model_name: str,
    estimator,
    X,
    y,
    splitter,
    colour: str = '#2563eb',
    leave: bool = False,
    show_fold_progress: bool = True,
) -> dict[str, np.ndarray]:
    y_array = np.asarray(y)
    splits = list(splitter.split(X, y_array))
    scores = {
        'test_accuracy': [],
        'test_macro_f1': [],
        'test_weighted_f1': [],
    }

    split_iterator = splits
    if show_fold_progress:
        split_iterator = progress_bar(
            splits,
            total=len(splits),
            desc=f'CV • {model_name}',
            colour=colour,
            leave=leave,
        )

    for train_indices, test_indices in split_iterator:
        fitted_model = clone(estimator)
        fitted_model.fit(index_rows(X, train_indices), index_rows(y_array, train_indices))
        y_true_fold = index_rows(y_array, test_indices)
        y_pred_fold = fitted_model.predict(index_rows(X, test_indices))

        fold_accuracy = accuracy_score(y_true_fold, y_pred_fold)
        fold_macro_f1 = f1_score(y_true_fold, y_pred_fold, average='macro', zero_division=0)
        fold_weighted_f1 = f1_score(y_true_fold, y_pred_fold, average='weighted', zero_division=0)

        scores['test_accuracy'].append(fold_accuracy)
        scores['test_macro_f1'].append(fold_macro_f1)
        scores['test_weighted_f1'].append(fold_weighted_f1)

        if show_fold_progress:
            split_iterator.set_postfix(acc=f'{fold_accuracy:.3f}', macro_f1=f'{fold_macro_f1:.3f}')

    return {metric_name: np.asarray(metric_values) for metric_name, metric_values in scores.items()}


def run_macro_f1_permutation_test(
    estimator,
    X,
    y,
    splitter,
    n_permutations: int,
    random_state: int,
) -> tuple[float, np.ndarray, float]:
    observed_scores = evaluate_cv_with_progress(
        'Observed Labels',
        estimator,
        X,
        y,
        splitter,
        colour='#f59e0b',
        leave=False,
        show_fold_progress=False,
    )
    observed_macro_f1 = float(np.mean(observed_scores['test_macro_f1']))
    rng = np.random.default_rng(random_state)
    y_array = np.asarray(y)
    permutation_scores: list[float] = []

    permutation_progress = progress_bar(
        range(1, n_permutations + 1),
        total=n_permutations,
        desc='Permutation Test',
        colour='#f59e0b',
    )
    for permutation_idx in permutation_progress:
        permuted_y = rng.permutation(y_array)
        null_scores = evaluate_cv_with_progress(
            f'Null {permutation_idx}',
            estimator,
            X,
            permuted_y,
            splitter,
            colour='#fbbf24',
            leave=False,
            show_fold_progress=False,
        )
        null_macro_f1 = float(np.mean(null_scores['test_macro_f1']))
        permutation_scores.append(null_macro_f1)
        current_pvalue = (sum(score >= observed_macro_f1 for score in permutation_scores) + 1) / (len(permutation_scores) + 1)
        permutation_progress.set_postfix(observed=f'{observed_macro_f1:.3f}', best_null=f'{max(permutation_scores):.3f}', p=f'{current_pvalue:.3f}')

    permutation_scores_array = np.asarray(permutation_scores)
    permutation_pvalue = float((np.sum(permutation_scores_array >= observed_macro_f1) + 1) / (n_permutations + 1))
    return observed_macro_f1, permutation_scores_array, permutation_pvalue


def run_learning_curve_with_progress(estimator, X, y, splitter, train_sizes: np.ndarray):
    y_array = np.asarray(y)
    splits = list(splitter.split(X, y_array))
    max_train_size = min(len(train_indices) for train_indices, _ in splits)
    train_sizes_array = np.asarray(train_sizes, dtype=float)

    if np.max(train_sizes_array) <= 1.0:
        absolute_train_sizes = np.unique(np.maximum(1, (train_sizes_array * max_train_size).astype(int)))
    else:
        absolute_train_sizes = np.unique(np.clip(train_sizes_array.astype(int), 1, max_train_size))

    train_score_rows: list[list[float]] = []
    valid_score_rows: list[list[float]] = []
    size_progress = progress_bar(
        absolute_train_sizes.tolist(),
        total=len(absolute_train_sizes),
        desc='Learning Curve',
        colour='#10b981',
    )

    for train_size in size_progress:
        fold_train_scores: list[float] = []
        fold_valid_scores: list[float] = []
        fold_progress = progress_bar(
            splits,
            total=len(splits),
            desc=f'Learning Curve • n={int(train_size)}',
            colour='#34d399',
            leave=False,
        )

        for train_indices, valid_indices in fold_progress:
            subset_train_indices = train_indices[:int(train_size)]
            fitted_model = clone(estimator)
            fitted_model.fit(index_rows(X, subset_train_indices), index_rows(y_array, subset_train_indices))

            y_train_true = index_rows(y_array, subset_train_indices)
            y_valid_true = index_rows(y_array, valid_indices)
            y_train_pred = fitted_model.predict(index_rows(X, subset_train_indices))
            y_valid_pred = fitted_model.predict(index_rows(X, valid_indices))

            train_macro_f1 = f1_score(y_train_true, y_train_pred, average='macro', zero_division=0)
            valid_macro_f1 = f1_score(y_valid_true, y_valid_pred, average='macro', zero_division=0)
            fold_train_scores.append(train_macro_f1)
            fold_valid_scores.append(valid_macro_f1)
            fold_progress.set_postfix(valid_f1=f'{valid_macro_f1:.3f}')

        train_score_rows.append(fold_train_scores)
        valid_score_rows.append(fold_valid_scores)
        size_progress.set_postfix(valid_f1=f'{np.mean(fold_valid_scores):.3f}')

    return absolute_train_sizes, np.asarray(train_score_rows), np.asarray(valid_score_rows)


print(f'Repo root: {REPO_ROOT}')
print(f'Runtime summary -> Colab: {IS_COLAB}, CUDA: {CUDA_AVAILABLE}, GPU: {GPU_NAME}')
if IS_COLAB and not CUDA_AVAILABLE:
    print('Warning: switch Colab runtime to T4 GPU for official full-dataset TwitterRoBERTa evaluation.')
else:
    print(f'TwitterRoBERTa embedding batch size: {ROBERTA_BATCH_SIZE}')

## 2. Load Dataset and Review QA Signals

In [ ]:
def build_text_hash(title: object, selftext: object) -> str:
    combined_text = f"{str(title).strip()} || {str(selftext).strip()}"
    return hashlib.sha256(combined_text.encode('utf-8')).hexdigest()


def compute_inline_summary(df: pd.DataFrame) -> pd.DataFrame:
    timestamp_series = pd.to_datetime(df['timestamp'], errors='coerce')
    rows = [
        {'section': 'rows', 'metric': 'rows_current', 'value': int(len(df))},
        {'section': 'duplicates', 'metric': 'duplicate_post_id_rows_current', 'value': int(df['post_id'].duplicated().sum())},
        {'section': 'duplicates', 'metric': 'duplicate_title_selftext_rows_current', 'value': int(df.duplicated(subset=['title', 'selftext']).sum())},
        {'section': 'missing', 'metric': 'missing_selftext_cleaned', 'value': int(df['selftext_cleaned'].isna().sum())},
        {'section': 'length', 'metric': 'word_count_min', 'value': int(df['word_count'].min())},
        {'section': 'length', 'metric': 'word_count_median', 'value': float(df['word_count'].median())},
        {'section': 'dates', 'metric': 'timestamp_min', 'value': timestamp_series.min().isoformat() if not timestamp_series.dropna().empty else ''},
        {'section': 'dates', 'metric': 'timestamp_max', 'value': timestamp_series.max().isoformat() if not timestamp_series.dropna().empty else ''},
    ]
    for label, count in df['label'].value_counts().sort_index().items():
        rows.append({'section': 'labels', 'metric': f'label_count_{label}', 'value': int(count)})
    return pd.DataFrame(rows)


df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=['selftext_cleaned']).reset_index(drop=True)

if 'text_hash' not in df.columns:
    df['text_hash'] = [build_text_hash(title, selftext) for title, selftext in zip(df['title'], df['selftext'])]

df['target'] = df['label'].map(LABEL_TO_TARGET)
assert df['target'].notna().all(), 'Unexpected label found in dataset.'

print(f'Dataset Size: {df.shape}')
print('\nLabel Distribution:')
display(df['label'].value_counts().rename_axis('label').reset_index(name='count'))

if DATASET_SUMMARY_PATH.exists():
    dataset_summary = pd.read_csv(DATASET_SUMMARY_PATH)
    print('\nSaved Dataset Summary:')
    display(dataset_summary)
else:
    dataset_summary = compute_inline_summary(df)
    print('\nInline Dataset Summary (dataset_summary.csv not found):')
    display(dataset_summary)

print('\nActive Duplicate Check On Modeling Input:')
print('duplicate post_id rows:', int(df['post_id'].duplicated().sum()))
print('duplicate title+selftext rows:', int(df.duplicated(subset=['title', 'selftext']).sum()))


## 3. Fixed Holdout Split and Repeated-CV Setup

We keep one deterministic 80/20 stratified split for demo confusion matrices and error analysis. Repeated cross-validation is used for stronger evidence in the results table.


In [ ]:
# Deterministic holdout split for demo plots and qualitative inspection
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['target'],
)

X_train_text = df.loc[train_idx, 'selftext_cleaned']
X_test_text = df.loc[test_idx, 'selftext_cleaned']
y_train = df.loc[train_idx, 'target']
y_test = df.loc[test_idx, 'target']

logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', multi_class='multinomial')),
])

linear_svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
])

sparse_models = {
    'TF-IDF + Logistic Regression': logreg_pipeline,
    'TF-IDF + LinearSVC': linear_svc_pipeline,
}


def summarize_cv_scores(model_name: str, cv_scores: dict[str, np.ndarray], dataset_scope: str) -> dict[str, object]:
    return {
        'model': model_name,
        'evaluation_type': 'repeated_cv',
        'dataset_scope': dataset_scope,
        'accuracy_mean': round(float(cv_scores['test_accuracy'].mean()), 4),
        'accuracy_std': round(float(cv_scores['test_accuracy'].std()), 4),
        'macro_f1_mean': round(float(cv_scores['test_macro_f1'].mean()), 4),
        'macro_f1_std': round(float(cv_scores['test_macro_f1'].std()), 4),
        'weighted_f1_mean': round(float(cv_scores['test_weighted_f1'].mean()), 4),
        'weighted_f1_std': round(float(cv_scores['test_weighted_f1'].std()), 4),
        'precision_control': np.nan,
        'recall_control': np.nan,
        'precision_moderate_mdd': np.nan,
        'recall_moderate_mdd': np.nan,
        'precision_severe_ideation': np.nan,
        'recall_severe_ideation': np.nan,
        'permutation_pvalue': np.nan,
        'notes': '5-fold, 3-repeat RepeatedStratifiedKFold',
    }


cv_rows = []
sparse_cv_progress = progress_bar(
    list(sparse_models.items()),
    total=len(sparse_models),
    desc='Sparse Repeated CV',
    colour='#7c3aed',
)
for model_name, estimator in sparse_cv_progress:
    cv_scores = evaluate_cv_with_progress(
        model_name,
        estimator,
        df['selftext_cleaned'],
        df['target'],
        RSKF,
        colour='#8b5cf6',
        leave=False,
    )
    cv_rows.append(summarize_cv_scores(model_name, cv_scores, 'full_processed_dataset'))
    sparse_cv_progress.set_postfix(model=model_name.split(' + ')[-1])

cv_results_df = pd.DataFrame(cv_rows).sort_values('macro_f1_mean', ascending=False)
print('Repeated Cross-Validation Results (mean ± std):')
display(cv_results_df[['model', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std', 'weighted_f1_mean', 'weighted_f1_std']])

## 4. Holdout Evaluation for Sparse Models

This section produces the demo confusion matrices plus per-class precision and recall for the fixed holdout split.


In [ ]:
def build_holdout_row(model_name: str, report: dict[str, dict[str, float]], accuracy: float, dataset_scope: str, notes: str = '') -> dict[str, object]:
    return {
        'model': model_name,
        'evaluation_type': 'holdout',
        'dataset_scope': dataset_scope,
        'accuracy_mean': round(float(accuracy), 4),
        'accuracy_std': np.nan,
        'macro_f1_mean': round(float(report['macro avg']['f1-score']), 4),
        'macro_f1_std': np.nan,
        'weighted_f1_mean': round(float(report['weighted avg']['f1-score']), 4),
        'weighted_f1_std': np.nan,
        'precision_control': round(float(report['Control']['precision']), 4),
        'recall_control': round(float(report['Control']['recall']), 4),
        'precision_moderate_mdd': round(float(report['Moderate MDD']['precision']), 4),
        'recall_moderate_mdd': round(float(report['Moderate MDD']['recall']), 4),
        'precision_severe_ideation': round(float(report['Severe Ideation']['precision']), 4),
        'recall_severe_ideation': round(float(report['Severe Ideation']['recall']), 4),
        'permutation_pvalue': np.nan,
        'notes': notes,
    }


holdout_rows = []
fitted_sparse_models = {}
logreg_error_frame = None
logreg_holdout_proba = None

sparse_holdout_progress = progress_bar(
    list(sparse_models.items()),
    total=len(sparse_models),
    desc='Sparse Holdout Models',
    colour='#22c55e',
)
for model_name, estimator in sparse_holdout_progress:
    fitted_model = clone(estimator)
    fitted_model.fit(X_train_text, y_train)
    y_pred = fitted_model.predict(X_test_text)

    report = classification_report(
        y_test,
        y_pred,
        target_names=LABEL_ORDER,
        output_dict=True,
        zero_division=0,
    )
    accuracy = accuracy_score(y_test, y_pred)
    holdout_rows.append(build_holdout_row(model_name, report, accuracy, 'full_processed_dataset', notes='Fixed 80/20 stratified holdout'))
    fitted_sparse_models[model_name] = fitted_model
    sparse_holdout_progress.set_postfix(acc=f'{accuracy:.3f}')

    print(f'\n{model_name} Accuracy: {accuracy:.4f}')
    print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=LABEL_ORDER, zero_division=0))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER)
    plt.title(f'{model_name} Holdout Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()

    if model_name == 'TF-IDF + Logistic Regression':
        logreg_holdout_proba = fitted_model.predict_proba(X_test_text)
        logreg_error_frame = df.loc[test_idx, ['post_id', 'title', 'selftext', 'selftext_cleaned', 'label', 'text_hash']].copy()
        logreg_error_frame['true_target'] = y_test.values
        logreg_error_frame['pred_target'] = y_pred
        logreg_error_frame['true_label'] = logreg_error_frame['true_target'].map(TARGET_TO_LABEL)
        logreg_error_frame['pred_label'] = logreg_error_frame['pred_target'].map(TARGET_TO_LABEL)
        logreg_error_frame['confidence'] = logreg_holdout_proba.max(axis=1)

holdout_results_df = pd.DataFrame(holdout_rows)
print('\nHoldout Metrics Summary:')
display(holdout_results_df[['model', 'accuracy_mean', 'macro_f1_mean', 'weighted_f1_mean', 'precision_severe_ideation', 'recall_severe_ideation']])


## 5. Permutation Test and Learning Curve for the Main Baseline

In [ ]:
permutation_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
permutation_score, permutation_scores, permutation_pvalue = run_macro_f1_permutation_test(
    logreg_pipeline,
    df['selftext_cleaned'],
    df['target'],
    permutation_cv,
    n_permutations=30,
    random_state=RANDOM_STATE,
)

print(f'Permutation Test Macro F1: {permutation_score:.4f}')
print(f'Permutation Test p-value: {permutation_pvalue:.4f}')

train_sizes, train_scores, valid_scores = run_learning_curve_with_progress(
    logreg_pipeline,
    df['selftext_cleaned'],
    df['target'],
    StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    train_sizes=np.linspace(0.2, 1.0, 5),
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train Macro F1')
plt.plot(train_sizes, valid_scores.mean(axis=1), marker='s', label='Validation Macro F1')
plt.fill_between(train_sizes, valid_scores.mean(axis=1) - valid_scores.std(axis=1), valid_scores.mean(axis=1) + valid_scores.std(axis=1), alpha=0.15)
plt.title('Learning Curve: TF-IDF + Logistic Regression')
plt.xlabel('Training Samples')
plt.ylabel('Macro F1 Score')
plt.legend()
plt.tight_layout()
plt.show()

## 6. TwitterRoBERTa Feature Extraction with Model-Card Preprocessing

The dense model now uses raw `selftext` with the model-card preprocessing for usernames and links. In Google Colab with a T4 GPU, the notebook keeps the full deduplicated dataset and uses a larger embedding batch size; CPU runs stay clearly labeled as fallback mode.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Accelerating TwitterRoBERTa Embeddings via: {gpu_name}')
print(f'TwitterRoBERTa embedding batch size: {ROBERTA_BATCH_SIZE}')

model_name = 'cardiffnlp/twitter-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)
bert_model.eval()

USER_PATTERN = re.compile(r'@\w+')
URL_PATTERN = re.compile(r'http\S+|www\.\S+')


def preprocess_for_twitter_roberta(text: object) -> str:
    normalized = str(text or '').strip()
    normalized = USER_PATTERN.sub('@user', normalized)
    normalized = URL_PATTERN.sub('http', normalized)
    return normalized


def autocast_context():
    if device.type == 'cuda':
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return nullcontext()


def get_batched_embeddings(texts: pd.Series, batch_size: int = ROBERTA_BATCH_SIZE) -> np.ndarray:
    all_embeddings = []
    start = 0
    batch_size = max(8, int(batch_size))
    embedding_progress = progress_bar(
        total=len(texts),
        desc='TwitterRoBERTa Embeddings',
        colour='#ec4899',
    )

    while start < len(texts):
        batch_texts = texts.iloc[start:start + batch_size].tolist()
        try:
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
            with torch.inference_mode():
                with autocast_context():
                    outputs = bert_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].float().cpu().numpy()
            all_embeddings.append(cls_embeddings)
            start += len(batch_texts)
            embedding_progress.update(len(batch_texts))
        except RuntimeError as exc:
            is_oom = 'out of memory' in str(exc).lower()
            if device.type == 'cuda' and is_oom and batch_size > 8:
                torch.cuda.empty_cache()
                batch_size = max(8, batch_size // 2)
                print(f'CUDA OOM encountered. Retrying with batch size {batch_size}.')
                continue
            raise

    embedding_progress.close()
    return np.vstack(all_embeddings)

full_roberta_df = df.copy()
full_roberta_df['roberta_text'] = full_roberta_df['selftext'].fillna('').apply(preprocess_for_twitter_roberta)

if torch.cuda.is_available():
    df_roberta_eval = full_roberta_df.copy()
    roberta_scope = 'full_processed_dataset'
    roberta_note = f'Official evaluation mode ({gpu_name}/full dataset)'
else:
    fallback_n = min(2000, len(full_roberta_df))
    df_roberta_eval = full_roberta_df.sample(n=fallback_n, random_state=RANDOM_STATE).reset_index(drop=True)
    roberta_scope = f'cpu_fallback_sample_{fallback_n}'
    roberta_note = 'Fallback mode: CPU sample used for practical local execution'

print(roberta_note)
print(f'RoBERTa evaluation rows: {len(df_roberta_eval)}')

X_bert = get_batched_embeddings(df_roberta_eval['roberta_text'])
y_bert = df_roberta_eval['target'].values

In [ ]:
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bert,
    y_bert,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_bert,
)

rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_training_progress = progress_bar(
    total=2,
    desc='TwitterRoBERTa + RF',
    colour='#ef4444',
)
rf_model.fit(X_train_b, y_train_b)
rf_training_progress.update(1)

y_pred_rf = rf_model.predict(X_test_b)
rf_report = classification_report(y_test_b, y_pred_rf, target_names=LABEL_ORDER, output_dict=True, zero_division=0)
rf_accuracy = accuracy_score(y_test_b, y_pred_rf)

print(f'TwitterRoBERTa + Random Forest Accuracy: {rf_accuracy:.4f}')
print('\nClassification Report:\n', classification_report(y_test_b, y_pred_rf, target_names=LABEL_ORDER, zero_division=0))

cm_rf = confusion_matrix(y_test_b, y_pred_rf, labels=[0, 1, 2])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER)
plt.title('TwitterRoBERTa + Random Forest Holdout Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

rf_cv_scores = evaluate_cv_with_progress(
    'TwitterRoBERTa + Random Forest',
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_bert,
    y_bert,
    RSKF,
    colour='#fb7185',
    leave=False,
)
rf_training_progress.update(1)
rf_training_progress.close()

cv_results_df = pd.concat([
    cv_results_df,
    pd.DataFrame([
        summarize_cv_scores('TwitterRoBERTa + Random Forest', rf_cv_scores, roberta_scope)
    ])
], ignore_index=True)

holdout_results_df = pd.concat([
    holdout_results_df,
    pd.DataFrame([
        build_holdout_row(
            'TwitterRoBERTa + Random Forest',
            rf_report,
            rf_accuracy,
            roberta_scope,
            notes=roberta_note,
        )
    ])
], ignore_index=True)

print('\nUpdated Cross-Validation Summary:')
display(cv_results_df[['model', 'dataset_scope', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std']])


## 7. SHAP and Token-Level Interpretation for the Main Sparse Baseline

In [ ]:
logreg_fitted = fitted_sparse_models['TF-IDF + Logistic Regression']
logreg_tfidf = logreg_fitted.named_steps['tfidf']
logreg_clf = logreg_fitted.named_steps['clf']

X_train_tfidf = logreg_tfidf.transform(X_train_text)
X_test_tfidf = logreg_tfidf.transform(X_test_text)
feature_names = np.array(logreg_tfidf.get_feature_names_out())

explainer_lr = shap.LinearExplainer(logreg_clf, X_train_tfidf, feature_names=feature_names)
subset_idx = min(100, X_test_tfidf.shape[0])
shap_values_lr = explainer_lr.shap_values(X_test_tfidf[:subset_idx])

print('SHAP Summary Plot for Severe Ideation (Class 2):')
shap.summary_plot(shap_values_lr[:, :, 2], X_test_tfidf[:subset_idx].toarray(), feature_names=feature_names)


def top_tokens_by_class(clf: LogisticRegression, feature_names_array: np.ndarray, top_n: int = 10) -> pd.DataFrame:
    rows = []
    for class_idx, class_label in TARGET_TO_LABEL.items():
        coef = clf.coef_[class_idx]
        top_indices = np.argsort(coef)[-top_n:][::-1]
        for rank, feature_idx in enumerate(top_indices, start=1):
            rows.append({
                'class_label': class_label,
                'rank': rank,
                'token': feature_names_array[feature_idx],
                'coefficient': round(float(coef[feature_idx]), 6),
            })
    return pd.DataFrame(rows)


top_tokens_df = top_tokens_by_class(logreg_clf, feature_names, top_n=10)
top_tokens_df.to_csv(TOP_TOKENS_PATH, index=False)
print('Top Influential Tokens By Class:')
display(top_tokens_df)

## 8. Error Analysis for the Main Baseline

We inspect representative holdout mistakes from the TF-IDF + Logistic Regression model and attach simple evidence-based reasons using influential token overlap.


In [ ]:
top_token_lookup = {
    class_label: set(top_tokens_df[top_tokens_df['class_label'] == class_label]['token'])
    for class_label in LABEL_ORDER
}


def infer_confusion_reason(row: pd.Series) -> str:
    tokens = set(str(row['selftext_cleaned']).split())
    supporting_terms = sorted(tokens.intersection(top_token_lookup[row['pred_label']]))[:5]
    if supporting_terms:
        return f"Predicted-class cue words present: {', '.join(supporting_terms)}"
    if row['true_label'] == 'Control' and row['pred_label'] != 'Control':
        return 'Negative emotional wording overlaps with distress-related vocabulary.'
    if row['true_label'] == 'Severe Ideation' and row['pred_label'] == 'Moderate MDD':
        return 'Distress is strong, but explicit ideation markers appear limited in the cleaned text.'
    if row['true_label'] == 'Moderate MDD' and row['pred_label'] == 'Severe Ideation':
        return 'The post contains high-intensity symptom or ideation-adjacent vocabulary.'
    return 'Shared depressive vocabulary across adjacent severity tiers likely caused the confusion.'


error_df = logreg_error_frame[logreg_error_frame['true_label'] != logreg_error_frame['pred_label']].copy()
error_df['possible_reason'] = error_df.apply(infer_confusion_reason, axis=1)
error_df['text_excerpt'] = error_df['selftext'].astype(str).str.slice(0, 280)
error_df = error_df.sort_values(['true_label', 'confidence'], ascending=[True, False])
error_df.to_csv(ERROR_ANALYSIS_PATH, index=False)

representative_errors = (
    error_df.groupby('true_label', group_keys=False)
    .head(3)
    [['post_id', 'title', 'true_label', 'pred_label', 'confidence', 'possible_reason', 'text_excerpt']]
)

print('Representative Holdout Errors (up to 3 per true class):')
display(representative_errors)

## 9. Exploratory Data Analysis and Language Patterns

In [ ]:
SYMPTOM_KEYWORDS = [
    'hopeless', 'worthless', 'suicide', 'suicidal', 'depressed', 'depression',
    'anxiety', 'anxious', 'lonely', 'loneliness', 'insomnia', 'fatigue',
    'guilt', 'guilty', 'numb', 'crying', 'empty', 'helpless',
    'selfharm', 'overdose', 'die', 'death', 'harm', 'pain'
]


def count_symptoms(text: object, keywords=SYMPTOM_KEYWORDS) -> int:
    if not isinstance(text, str):
        return 0
    tokens = text.lower().split()
    return sum(1 for token in tokens if token in keywords)


df['symptom_count'] = df['selftext_cleaned'].apply(count_symptoms)

fig, axes = plt.subplots(2, 1, figsize=(18, 10))
class_means = df.groupby('label')['symptom_count'].mean().reindex(LABEL_ORDER)
class_means.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black')
axes[0].set_title('Average Symptom Keyword Count by Class', fontsize=14)
axes[0].set_ylabel('Mean Count per Post')
axes[0].tick_params(axis='x', rotation=0)

keyword_freq = pd.DataFrame({
    kw: [
        df[df['label'] == label]['selftext_cleaned'].str.count(kw).sum()
        for label in LABEL_ORDER
    ]
    for kw in SYMPTOM_KEYWORDS
}, index=LABEL_ORDER)
keyword_freq_log = np.log1p(keyword_freq)

sns.heatmap(
    keyword_freq_log,
    annot=keyword_freq.values,
    fmt='g',
    cmap='YlOrRd',
    linewidths=0.6,
    linecolor='white',
    ax=axes[1],
    cbar_kws={'label': 'Log-Scaled Frequency'},
    annot_kws={'size': 9, 'weight': 'bold'},
)
axes[1].set_title('Symptom Keyword Frequency Heatmap', fontsize=14)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = {'Control': 'Greens', 'Moderate MDD': 'Oranges', 'Severe Ideation': 'Reds'}

for idx, label in enumerate(LABEL_ORDER):
    text = ' '.join(df[df['label'] == label]['selftext_cleaned'].dropna())
    wc = WordCloud(width=800, height=400, background_color='white', colormap=colors[label], max_words=150).generate(text)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(label, fontsize=15)
    axes[idx].axis('off')

plt.suptitle('Word Clouds by Severity Tier', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.kdeplot(data=df, x='sentiment_score', hue='label', common_norm=False, fill=True, ax=axes[0], hue_order=LABEL_ORDER)
axes[0].set_title('VADER Sentiment Score Distribution')
axes[0].set_xlabel('Compound Sentiment Score')

sns.boxplot(data=df, x='label', y='word_count', order=LABEL_ORDER, palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[1])
axes[1].set_title('Post Length Distribution by Class')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Word Count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
vectorizer = CountVectorizer(ngram_range=(2, 2), max_features=15)
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

for idx, label in enumerate(LABEL_ORDER):
    subset = df[df['label'] == label]['selftext_cleaned']
    X_bi = vectorizer.fit_transform(subset)
    counts = np.asarray(X_bi.sum(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    top_idx = np.argsort(counts)[::-1]

    sns.barplot(
        x=counts[top_idx],
        y=terms[top_idx],
        palette='magma',
        ax=axes[idx],
    )
    axes[idx].set_title(f'Top Bigrams: {label}')
    axes[idx].set_xlabel('Frequency')
    axes[idx].set_ylabel('')

plt.tight_layout()
plt.show()

## 10. Export Summary Artifacts

In [ ]:
permutation_row = pd.DataFrame([
    {
        'model': 'TF-IDF + Logistic Regression',
        'evaluation_type': 'permutation_test',
        'dataset_scope': 'full_processed_dataset',
        'accuracy_mean': np.nan,
        'accuracy_std': np.nan,
        'macro_f1_mean': round(float(permutation_score), 4),
        'macro_f1_std': np.nan,
        'weighted_f1_mean': np.nan,
        'weighted_f1_std': np.nan,
        'precision_control': np.nan,
        'recall_control': np.nan,
        'precision_moderate_mdd': np.nan,
        'recall_moderate_mdd': np.nan,
        'precision_severe_ideation': np.nan,
        'recall_severe_ideation': np.nan,
        'permutation_pvalue': round(float(permutation_pvalue), 6),
        'notes': 'Permutation test on macro F1 for TF-IDF + Logistic Regression',
    }
])

results_summary_df = pd.concat([cv_results_df, holdout_results_df, permutation_row], ignore_index=True)
results_summary_df.to_csv(RESULTS_SUMMARY_PATH, index=False)

print(f'Saved results summary to: {RESULTS_SUMMARY_PATH}')
print(f'Saved error analysis to: {ERROR_ANALYSIS_PATH}')
print(f'Saved top tokens to: {TOP_TOKENS_PATH}')
if IS_COLAB:
    print('Colab note: exported CSV artifacts are now inside the cloned repo under data/processed/.')
display(results_summary_df)

## 11. GitHub Repo Sync From Colab

This cell stages the full repo state from the Colab clone and pushes it back to GitHub by default when `GITHUB_TOKEN` is present.


In [ ]:
import subprocess

ARTIFACT_PATHS = [DATASET_SUMMARY_PATH, RESULTS_SUMMARY_PATH, ERROR_ANALYSIS_PATH, TOP_TOKENS_PATH]
generated_artifacts = [path for path in ARTIFACT_PATHS if path.exists()]
artifact_rows = [str(path.relative_to(REPO_ROOT)) for path in generated_artifacts]
display(pd.DataFrame({'artifact_path': artifact_rows}))

AUTO_PUSH_CHANGES = True
COMMIT_MESSAGE = f"Sync Colab notebook changes ({pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M UTC')})"

if not IS_COLAB:
    print('Local run detected. Skip GitHub repo sync here.')
elif not github_token:
    print('No GITHUB_TOKEN found. Add the secret and rerun from the top if you want push access.')
elif not git_user_name or not git_user_email:
    print('Git identity is missing. Configure user.name and user.email before pushing from Colab.')
else:
    status_result = subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'status', '--short'],
        capture_output=True,
        text=True,
        check=True,
    )
    repo_status = status_result.stdout.strip()
    print('Current repo status:')
    print(repo_status or 'Working tree is currently clean.')

    if AUTO_PUSH_CHANGES:
        branch_name = subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'rev-parse', '--abbrev-ref', 'HEAD'],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()

        subprocess.run(['git', '-C', str(REPO_ROOT), 'add', '-A'], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'diff', '--cached', '--quiet'],
            check=False,
        )

        if diff_result.returncode == 0:
            print('No staged changes detected after git add -A. Nothing to commit.')
        else:
            subprocess.run(['git', '-C', str(REPO_ROOT), 'commit', '-m', COMMIT_MESSAGE], check=True)
            subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--rebase', 'origin', branch_name], check=True)
            subprocess.run(['git', '-C', str(REPO_ROOT), 'push', 'origin', branch_name], check=True)
            print(f'Pushed all staged repo changes to origin/{branch_name}')
    else:
        print('Set AUTO_PUSH_CHANGES = True in this cell if you want to commit and push the current Colab repo state.')


## 12. Dataset Card and Limitations

### Intended Use
This project is designed for **academic analysis in a Big Data Analytics course**. It studies linguistic patterns in Reddit posts and compares NLP pipelines for severity-oriented text classification.

### Data Source and Scope
- Source: Reddit submissions collected through PullPush for `r/depression`, `r/SuicideWatch`, and `r/CasualConversation`
- Labels are **proxy labels** derived from subreddit origin, not clinical diagnoses.
- The current processed snapshot may differ slightly from the original raw extraction target after QA filtering and minimum-length filtering.

### Important Limitations
- **Self-report is not diagnosis**: subreddit membership or posting context does not confirm Major Depressive Disorder.
- **Label noise exists**: some posts may not fit their class cleanly, especially between Moderate MDD and Severe Ideation.
- **Privacy and ethics matter**: these posts come from sensitive contexts. Outputs should be used carefully and discussed respectfully.
- **Not for clinical deployment**: this notebook is for coursework, exploratory analysis, and methodology comparison only, not for real-world mental-health triage.
- **Transformer fallback mode**: local CPU runs may use a smaller sample for practical runtime; GPU/Colab is the preferred path for official transformer evaluation.
